# 🌿 Milán Verde & Sostenible — Análisis Exploratorio
**Fuentes:** Verde Urbano, Consumo de recursos, Días calurosos por zona, Techos verdes, Zonas sin metano, m² por habitante  
**Ciudad:** Milán, Italia

---
## Estructura del notebook
1. Setup y carga de datos
2. Verde Urbano (6 gráficos)
3. Consumo de Recursos (5 gráficos)
4. Clima por Zona (5 gráficos)
5. Techos Verdes (5 gráficos)
6. Zonas sin Metano (5 gráficos)
7. Análisis Cruzado / Correlaciones (4 gráficos)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Estilo global ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1a0f',
    'axes.facecolor':   '#111811',
    'axes.edgecolor':   '#1e2e1e',
    'axes.labelcolor':  '#6b8f6b',
    'xtick.color':      '#6b8f6b',
    'ytick.color':      '#6b8f6b',
    'text.color':       '#e8f5e8',
    'grid.color':       '#1e2e1e',
    'grid.linestyle':   '--',
    'grid.alpha':       0.6,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   14,
    'axes.titlecolor':  '#e8f5e8',
    'axes.titlepad':    14,
    'axes.titleweight': 'bold',
    'legend.facecolor': '#111811',
    'legend.edgecolor': '#1e2e1e',
    'legend.labelcolor':'#6b8f6b',
})

# ── Paletas ────────────────────────────────────────────────────────────────────
GREENS  = ['#4ade80','#22c55e','#16a34a','#a3e635','#84cc16','#65a30d','#4d7c0f','#365314']
WARM    = ['#fbbf24','#f59e0b','#d97706','#b45309','#92400e']
BLUES   = ['#60a5fa','#3b82f6','#2563eb','#1d4ed8','#1e40af']
MIXED   = GREENS[:4] + WARM[:3] + BLUES[:3]
RED     = '#f87171'
ACCENT  = '#a3e635'

def savefig(name):
    plt.tight_layout()
    plt.show()

print('✅ Setup completo')

In [ ]:
# ── Carga de datos ─────────────────────────────────────────────────────────────
verde_raw   = pd.read_csv('Verde_Urbano.csv')
mq_hab      = pd.read_csv('Metros_cuadrados_habitante.csv')
consumo_raw = pd.read_csv('Consumo_agua_gas_electricidad_2011.csv')
techos_raw  = pd.read_csv('techos_verdes_Milan.csv')
metano_raw  = pd.read_csv('zonas_sin_metano_Milan.csv', sep=';')

# Clima manual (semicolon, transpuesta)
clima_data = {
    'zona':        ['Bicocca','Bocconi','Bovisa','Centro','Città Studi','San Siro','Sur'],
    'temp_media':  [15.4,15.7,15.4,15.8,15.4,15.0,15.2],
    'temp_max':    [38.5,37.6,38.4,38.2,38.1,37.5,37.5],
    'temp_min':    [-3.9,-3.5,-3.7,-3.6,-4.2,-4.6,-5.0],
    'dias_gelo':   [11.8,9.2,14.0,7.2,10.2,16.3,18.0],
    'dias_calura': [56.3,52.3,55.2,52.8,45.8,48.5,50.0],
    'noches_trop': [55.5,63.3,56.3,62.3,58.5,54.5,56.8],
    'humedad':     [60.8,59.1,59.4,59.1,59.7,60.7,61.7],
    'precip_mm':   [1017.9,906.4,1139.0,1016.9,1056.8,734.0,873.5],
    'dias_lluvia': [84.0,79.2,83.3,85.2,84.0,76.3,80.0],
    'viento_ms':   [1.2,1.5,1.0,1.3,1.6,1.3,1.1],
}
clima = pd.DataFrame(clima_data)

# Limpiar columnas
verde_raw.columns = verde_raw.columns.str.strip()
consumo_raw.columns = consumo_raw.columns.str.strip()
techos_raw['Classifica'] = techos_raw['Classifica'].str.strip()
verde_raw['Tipologia di Area (valore in MQ)'] = verde_raw['Tipologia di Area (valore in MQ)'].str.strip()

# Alias cortos
tipo_alias = {
    'Parchi (giardini e ville) urbani': 'Parchi urbani',
    'Verde attrezzato': 'Verde attrezzato',
    'Aree di arredo urbano': 'Arredo urbano',
    'Giardini scolastici comunali': 'Giar. scolastici',
    'Ville giardini e parchi': 'Ville/Parchi',
    'Cimiteri': 'Cimiteri',
    'Altre tipologie di verde urbano': 'Altre',
    'Forestazione urbana': 'Forestazione',
    'Orti botanici': 'Orti botanici',
    "Aree all'aperto sportive e a servizio ludico ricreativo": 'Sport/Ludico',
    'Orti urbani': 'Orti urbani',
    'Aree naturali protette': 'Aree naturali',
}
verde_raw['tipo_short'] = verde_raw['Tipologia di Area (valore in MQ)'].map(tipo_alias).fillna(verde_raw['Tipologia di Area (valore in MQ)'])

consumo_tipo = {
    'Energia elettrica per uso domestico': 'Electricidad (kWh)',
    'Gas metano per uso domestico e riscaldamento': 'Gas metano (m³)',
    'Acqua fatturata per uso domestico': 'Agua (m³)',
}
consumo_raw['tipo_es'] = consumo_raw['Consumo pro capite tipo'].map(consumo_tipo)

print('Verde Urbano:',    verde_raw.shape)
print('m² / habitante:', mq_hab.shape)
print('Consumo:',         consumo_raw.shape)
print('Techos verdes:',   techos_raw.shape)
print('Metano:',          metano_raw.shape)
print('Clima:',           clima.shape)
verde_raw.head(3)

## 🌳 Sección 1 — Verde Urbano

In [ ]:
# ── Gráfico 1.1 · Línea: m² de verde por habitante 2011–2024 ──────────────────
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(mq_hab['Anno'], mq_hab['Mq_verde_urbano_per_abitante'],
        color=GREENS[0], lw=2.5, marker='o', markersize=7,
        markerfacecolor='#0f1a0f', markeredgecolor=GREENS[0], markeredgewidth=2)

ax.fill_between(mq_hab['Anno'], mq_hab['Mq_verde_urbano_per_abitante'],
                mq_hab['Mq_verde_urbano_per_abitante'].min() - 0.3,
                alpha=0.15, color=GREENS[0])

# Anotaciones min/max
imax = mq_hab['Mq_verde_urbano_per_abitante'].idxmax()
imin = mq_hab['Mq_verde_urbano_per_abitante'].idxmin()
for i, label, dy in [(imax, 'Máx\n18.9 m²', 0.15), (imin, 'Mín\n17.2 m²', -0.3)]:
    ax.annotate(label, xy=(mq_hab.loc[i,'Anno'], mq_hab.loc[i,'Mq_verde_urbano_per_abitante']),
                xytext=(0, 28 if dy>0 else -38), textcoords='offset points',
                ha='center', fontsize=9, color=ACCENT,
                arrowprops=dict(arrowstyle='->', color=ACCENT, lw=1.2))

ax.set_title('Verde urbano por habitante en Milán (2011–2024)')
ax.set_xlabel('Año'); ax.set_ylabel('m² / habitante')
ax.set_ylim(16.8, 19.5)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
ax.grid(True, axis='y')
savefig('verde_linea_mq_habitante')

In [ ]:
# ── Gráfico 1.2 · Barras apiladas: total m² por tipo (2011–2024) ──────────────
verde_total = verde_raw.groupby(['Anno','tipo_short'])['Metri Quadri'].sum().unstack(fill_value=0)
tipos_sorted = verde_total.sum().sort_values(ascending=False).index.tolist()

fig, ax = plt.subplots(figsize=(14, 6))
bottom = np.zeros(len(verde_total))
palette = GREENS + WARM + BLUES

for i, tipo in enumerate(tipos_sorted):
    if tipo in verde_total.columns:
        vals = verde_total[tipo].values / 1e6
        bars = ax.bar(verde_total.index, vals, bottom=bottom,
                      color=palette[i % len(palette)], label=tipo,
                      edgecolor='#0f1a0f', linewidth=0.4)
        bottom += vals

ax.set_title('Composición total de verde urbano por tipo (2011–2024)')
ax.set_xlabel('Año'); ax.set_ylabel('Millones de m²')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f M'))
savefig('verde_barras_apiladas_tipo')

In [ ]:
# ── Gráfico 1.3 · Donut: distribución por tipo (último año disponible) ─────────
ultimo = verde_raw['Anno'].max()
df_ult = verde_raw[verde_raw['Anno'] == ultimo].groupby('tipo_short')['Metri Quadri'].sum()
df_ult = df_ult[df_ult > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 7))
wedges, texts, autotexts = ax.pie(
    df_ult.values, labels=None,
    autopct=lambda p: f'{p:.1f}%' if p > 3 else '',
    colors=GREENS + WARM[:4],
    startangle=140, pctdistance=0.78,
    wedgeprops=dict(width=0.52, edgecolor='#0f1a0f', linewidth=1.5))

for at in autotexts: at.set(color='#0f1a0f', fontsize=9, fontweight='bold')

ax.legend(wedges, df_ult.index, loc='lower right', fontsize=9,
          bbox_to_anchor=(1.25, 0.05))
ax.set_title(f'Distribución de verde urbano por tipo ({ultimo})')
savefig('verde_donut_distribucion')

In [ ]:
# ── Gráfico 1.4 · Small multiples: evolución por tipo ─────────────────────────
top_tipos = verde_total.sum().nlargest(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Evolución por tipo de área verde (2011–2024)', fontsize=15, color='#e8f5e8', y=1.01)

for ax, tipo, color in zip(axes.flat, top_tipos, GREENS):
    vals = verde_total[tipo] / 1e6
    ax.plot(verde_total.index, vals, color=color, lw=2, marker='o', markersize=4)
    ax.fill_between(verde_total.index, vals, vals.min()*0.98, alpha=0.15, color=color)
    ax.set_title(tipo, fontsize=10)
    ax.set_xlabel(''); ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f M'))
    ax.grid(True, axis='y', alpha=0.4)

savefig('verde_small_multiples')

In [ ]:
# ── Gráfico 1.5 · Heatmap: m² por tipo y año ──────────────────────────────────
pivot = verde_total[top_tipos].T / 1e6

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, ax=ax, cmap='YlGn', linewidths=0.4, linecolor='#0f1a0f',
            annot=True, fmt='.1f', annot_kws={'size':8, 'color':'#0f1a0f'},
            cbar_kws={'label': 'Millones de m²'})

ax.set_title('Heatmap: m² de verde por tipo y año (millones)')
ax.set_xlabel('Año'); ax.set_ylabel('')
ax.tick_params(axis='x', rotation=45)
savefig('verde_heatmap_tipo_anno')

In [ ]:
# ── Gráfico 1.6 · Barras horizontales: variación % 2011 → último año ──────────
v2011 = verde_raw[verde_raw['Anno']=='2011'].groupby('tipo_short')['Metri Quadri'].sum()
vult  = verde_raw[verde_raw['Anno']==ultimo].groupby('tipo_short')['Metri Quadri'].sum()
comun = v2011.index.intersection(vult.index)
delta = ((vult[comun] - v2011[comun]) / v2011[comun].replace(0, np.nan) * 100).dropna().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = [GREENS[1] if v >= 0 else RED for v in delta.values]
bars = ax.barh(delta.index, delta.values, color=colors, edgecolor='#0f1a0f', height=0.6)

for bar, val in zip(bars, delta.values):
    ax.text(val + (1 if val >= 0 else -1), bar.get_y() + bar.get_height()/2,
            f'{val:+.1f}%', va='center', ha='left' if val >= 0 else 'right',
            fontsize=9, color='#e8f5e8')

ax.axvline(0, color='#6b8f6b', lw=0.8)
ax.set_title(f'Variación % por tipo de verde: 2011 → {ultimo}')
ax.set_xlabel('Variación (%)')
ax.grid(True, axis='x', alpha=0.4)
savefig('verde_barras_variacion_pct')

## ⚡ Sección 2 — Consumo de Recursos per cápita

In [ ]:
# ── Gráfico 2.1 · Líneas: evolución de los 3 recursos ─────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle('Consumo per cápita en Milán (2000–2011)', fontsize=15, color='#e8f5e8')

configs = [
    ('Electricidad (kWh)', WARM[0], 'kWh/hab'),
    ('Gas metano (m³)',    RED,     'm³/hab'),
    ('Agua (m³)',          BLUES[0],'m³/hab'),
]
for ax, (tipo, color, unit) in zip(axes, configs):
    df = consumo_raw[consumo_raw['tipo_es']==tipo].sort_values('anno')
    ax.plot(df['anno'], df['Consumo pro capite'], color=color, lw=2.5,
            marker='o', markersize=6, markerfacecolor='#111811', markeredgewidth=2)
    ax.fill_between(df['anno'], df['Consumo pro capite'],
                    df['Consumo pro capite'].min() * 0.97, alpha=0.12, color=color)
    ax.set_ylabel(unit); ax.set_title(tipo, fontsize=11)
    ax.grid(True, axis='y', alpha=0.4)
    # Trend line
    z = np.polyfit(df['anno'], df['Consumo pro capite'], 1)
    p = np.poly1d(z)
    ax.plot(df['anno'], p(df['anno']), '--', color=color, alpha=0.4, lw=1.2, label='Tendencia')

axes[-1].set_xlabel('Año')
savefig('consumo_lineas_3recursos')

In [ ]:
# ── Gráfico 2.2 · Normalizado base 100 = año 2000 ─────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

colores = [WARM[0], RED, BLUES[0]]
for tipo, color in zip(['Electricidad (kWh)','Gas metano (m³)','Agua (m³)'], colores):
    df = consumo_raw[consumo_raw['tipo_es']==tipo].sort_values('anno').copy()
    base = df[df['anno']==2000]['Consumo pro capite'].values[0]
    df['norm'] = df['Consumo pro capite'] / base * 100
    ax.plot(df['anno'], df['norm'], color=color, lw=2.2, marker='o',
            markersize=5, label=tipo.split(' (')[0])

ax.axhline(100, color='#6b8f6b', lw=0.8, ls='--', label='Base 2000 = 100')
ax.fill_between([2000,2011], 70, 100, alpha=0.04, color=GREENS[0])
ax.set_title('Consumo per cápita normalizado (Base 100 = año 2000)')
ax.set_xlabel('Año'); ax.set_ylabel('Índice (2000=100)')
ax.set_ylim(70, 115)
ax.legend(); ax.grid(True, axis='y', alpha=0.4)
savefig('consumo_normalizado_base100')

In [ ]:
# ── Gráfico 2.3 · Barras agrupadas: 2000 vs 2011 comparativa ──────────────────
tipos = ['Electricidad (kWh)','Gas metano (m³)','Agua (m³)']
v2000 = [consumo_raw[(consumo_raw['tipo_es']==t)&(consumo_raw['anno']==2000)]['Consumo pro capite'].values[0] for t in tipos]
v2011 = [consumo_raw[(consumo_raw['tipo_es']==t)&(consumo_raw['anno']==2011)]['Consumo pro capite'].values[0] for t in tipos]

x = np.arange(3); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, v2000, w, label='2000', color=[WARM[2], RED, BLUES[2]], alpha=0.6, edgecolor='#0f1a0f')
b2 = ax.bar(x + w/2, v2011, w, label='2011', color=[WARM[0], RED, BLUES[0]], edgecolor='#0f1a0f')

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{bar.get_height():.0f}', ha='center', fontsize=9, color='#e8f5e8')

ax.set_xticks(x)
ax.set_xticklabels([t.split(' (')[0] for t in tipos])
ax.set_title('Comparativa consumo per cápita: 2000 vs 2011')
ax.set_ylabel('Consumo (unidad propia)')
ax.legend(); ax.grid(True, axis='y', alpha=0.4)
savefig('consumo_barras_2000_vs_2011')

In [ ]:
# ── Gráfico 2.4 · Waterfall: reducción absoluta gas metano ────────────────────
gas = consumo_raw[consumo_raw['tipo_es']=='Gas metano (m³)'].sort_values('anno').copy()
gas['delta'] = gas['Consumo pro capite'].diff()
gas = gas.dropna()

fig, ax = plt.subplots(figsize=(12, 5))
running = gas.iloc[0]['Consumo pro capite'] + gas.iloc[0]['delta']
bottoms = []
for i, row in gas.iterrows():
    d = row['delta']
    b = running - d if d < 0 else running
    bottoms.append(min(running, running - d))
    running = running

# Simple bar chart of deltas
colors = [GREENS[1] if d < 0 else RED for d in gas['delta']]
ax.bar(gas['anno'], gas['delta'], color=colors, edgecolor='#0f1a0f', width=0.7)
ax.axhline(0, color='#6b8f6b', lw=0.8)
ax.set_title('Variación anual del consumo de gas metano per cápita (m³/hab)')
ax.set_xlabel('Año'); ax.set_ylabel('Δ m³/hab')
ax.grid(True, axis='y', alpha=0.4)

total_delta = gas['delta'].sum()
ax.text(0.98, 0.05, f'Total acumulado: {total_delta:+.1f} m³/hab',
        transform=ax.transAxes, ha='right', fontsize=10,
        color=GREENS[0] if total_delta < 0 else RED,
        bbox=dict(boxstyle='round', facecolor='#111811', edgecolor='#1e2e1e', alpha=0.8))
savefig('consumo_waterfall_gas')

In [ ]:
# ── Gráfico 2.5 · Área: todos los recursos en un solo panel ───────────────────
fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

for tipo, color, ax in [('Gas metano (m³)', RED, ax1), ('Agua (m³)', BLUES[0], ax1),
                         ('Electricidad (kWh)', WARM[0], ax2)]:
    df = consumo_raw[consumo_raw['tipo_es']==tipo].sort_values('anno')
    (ax2 if tipo=='Electricidad (kWh)' else ax1).plot(
        df['anno'], df['Consumo pro capite'],
        color=color, lw=2.2, label=tipo, marker='o', markersize=4)

ax1.set_ylabel('Gas / Agua (m³/hab)', color='#6b8f6b')
ax2.set_ylabel('Electricidad (kWh/hab)', color=WARM[0])
ax1.set_xlabel('Año')
ax1.set_title('Todos los recursos en una vista (2000–2011)')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper right')
ax1.grid(True, axis='y', alpha=0.3)
savefig('consumo_dual_axis')

## 🌡️ Sección 3 — Clima por Zona

In [ ]:
# ── Gráfico 3.1 · Barras: temperatura media por zona ──────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
norm = plt.Normalize(clima['temp_media'].min(), clima['temp_media'].max())
cmap = plt.cm.RdYlGn_r
colors = [cmap(norm(v)) for v in clima['temp_media']]

bars = ax.bar(clima['zona'], clima['temp_media'], color=colors, edgecolor='#0f1a0f', width=0.6)
for bar, val in zip(bars, clima['temp_media']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val}°C', ha='center', fontsize=10, color='#e8f5e8')

ax.set_title('Temperatura media anual por zona de Milán (°C)')
ax.set_ylabel('°C'); ax.set_ylim(14.5, 16.3)
ax.grid(True, axis='y', alpha=0.4)
savefig('clima_temp_media_zona')

In [ ]:
# ── Gráfico 3.2 · Grouped bars: días calurosos vs noches tropicales ────────────
x = np.arange(len(clima)); w = 0.35
fig, ax = plt.subplots(figsize=(12, 5))

b1 = ax.bar(x - w/2, clima['dias_calura'], w, label='Días de calor', color=WARM[0], edgecolor='#0f1a0f')
b2 = ax.bar(x + w/2, clima['noches_trop'], w, label='Noches tropicales', color=RED, edgecolor='#0f1a0f')

ax.set_xticks(x); ax.set_xticklabels(clima['zona'])
ax.set_title('Días de calor vs Noches tropicales por zona')
ax.set_ylabel('N° promedio anual de días'); ax.legend()
ax.grid(True, axis='y', alpha=0.4)
savefig('clima_calor_vs_noches_tropicales')

In [ ]:
# ── Gráfico 3.3 · Heatmap completo de variables climáticas ────────────────────
clima_hm = clima.set_index('zona')[['temp_media','temp_max','temp_min',
                                    'dias_calura','noches_trop','dias_gelo',
                                    'humedad','precip_mm','dias_lluvia','viento_ms']]
climate_norm = (clima_hm - clima_hm.min()) / (clima_hm.max() - clima_hm.min())

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(climate_norm.T, ax=ax, cmap='YlGn', linewidths=0.4, linecolor='#0f1a0f',
            annot=clima_hm.T, fmt='.1f', annot_kws={'size': 8, 'color':'#0f1a0f'},
            cbar_kws={'label': 'Valor normalizado'})
ax.set_title('Heatmap de indicadores climáticos por zona (valores reales anotados)')
ax.set_xlabel('Zona'); ax.set_ylabel('')
savefig('clima_heatmap_completo')

In [ ]:
# ── Gráfico 3.4 · Radar/Spider: perfil climático por zona ─────────────────────
from matplotlib.patches import FancyArrowPatch
metrics = ['temp_media','dias_calura','noches_trop','dias_gelo','humedad','precip_mm']
metric_labels = ['T. media','Días calor','Noches trop.','Días hielo','Humedad','Precip.']
N = len(metrics)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

# Normalizar 0–1
norm_df = (clima[metrics] - clima[metrics].min()) / (clima[metrics].max() - clima[metrics].min())

fig, axes = plt.subplots(2, 4, figsize=(16, 8), subplot_kw=dict(polar=True))
fig.suptitle('Perfil climático por zona de Milán', fontsize=14, color='#e8f5e8')

for i, (ax, (_, row)) in enumerate(zip(axes.flat[:7], norm_df.iterrows())):
    values = row[metrics].tolist() + [row[metrics[0]]]
    ax.plot(angles, values, color=GREENS[i % len(GREENS)], lw=2)
    ax.fill(angles, values, color=GREENS[i % len(GREENS)], alpha=0.2)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metric_labels, fontsize=8, color='#6b8f6b')
    ax.set_yticks([])
    ax.set_title(clima.loc[i,'zona'], fontsize=10, color='#e8f5e8', pad=12)
    ax.set_facecolor('#111811')
    ax.grid(color='#1e2e1e', alpha=0.6)

axes.flat[7].set_visible(False)
savefig('clima_radar_zonas')

In [ ]:
# ── Gráfico 3.5 · Scatter: precipitación vs humedad por zona ──────────────────
fig, ax = plt.subplots(figsize=(9, 6))

scatter = ax.scatter(clima['precip_mm'], clima['humedad'],
                     c=clima['temp_media'], cmap='RdYlGn_r',
                     s=clima['dias_calura'] * 5,
                     edgecolors='#e8f5e8', linewidths=0.6, alpha=0.85)

for _, row in clima.iterrows():
    ax.annotate(row['zona'], (row['precip_mm'], row['humedad']),
                textcoords='offset points', xytext=(6, 4), fontsize=8, color='#e8f5e8')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Temp. media (°C)', color='#6b8f6b')
ax.set_xlabel('Precipitaciones totales (mm)')
ax.set_ylabel('Humedad relativa (%)')
ax.set_title('Precipitación vs Humedad por zona\n(tamaño = días de calor, color = temp. media)')
ax.grid(True, alpha=0.3)
savefig('clima_scatter_precip_humedad')

## 🏢 Sección 4 — Techos Verdes

In [ ]:
# ── Preparar datos techos ──────────────────────────────────────────────────────
techo_alias = {
    'Insediamenti residenziali': 'Residencial',
    'Insediamenti industriali, artigianali, commerciali': 'Industrial/Comercial',
    'Servizi pubblici': 'Servicios públicos',
    'Edifici scolastici': 'Escuelas',
    'Insediamenti ospedalieri': 'Hospitales',
    'Universit e ricerca': 'Universidad/Invest.',
    'Sport': 'Deporte',
    'Strutture ricettive': 'Hospedaje',
}
techos_raw['tipo_es'] = techos_raw['Classifica'].map(techo_alias).fillna(techos_raw['Classifica'])

# ── Gráfico 4.1 · Barras horizontales: conteo por tipo ────────────────────────
conteo = techos_raw['tipo_es'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(conteo.index, conteo.values,
               color=GREENS[:len(conteo)], edgecolor='#0f1a0f', height=0.6)

for bar, val in zip(bars, conteo.values):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{val:,}  ({val/len(techos_raw)*100:.1f}%)',
            va='center', fontsize=9, color='#e8f5e8')

ax.set_title('Techos verdes por tipo de edificio en Milán')
ax.set_xlabel('Número de edificios')
ax.grid(True, axis='x', alpha=0.4)
savefig('techos_barras_tipo')

In [ ]:
# ── Gráfico 4.2 · Box plot: distribución de área por tipo ─────────────────────
tipos_orden = techos_raw.groupby('tipo_es')['area'].median().sort_values(ascending=False).index.tolist()

fig, ax = plt.subplots(figsize=(13, 6))
data_bp = [techos_raw[techos_raw['tipo_es']==t]['area'].dropna().values for t in tipos_orden]

bp = ax.boxplot(data_bp, patch_artist=True, vert=True,
                medianprops=dict(color='#0f1a0f', lw=2),
                whiskerprops=dict(color='#6b8f6b'),
                capprops=dict(color='#6b8f6b'),
                flierprops=dict(marker='.', color='#6b8f6b', markersize=3, alpha=0.4))

for patch, color in zip(bp['boxes'], GREENS[:len(tipos_orden)]):
    patch.set_facecolor(color); patch.set_alpha(0.75)

ax.set_xticks(range(1, len(tipos_orden)+1))
ax.set_xticklabels(tipos_orden, rotation=20, ha='right')
ax.set_title('Distribución del área de techo verde por tipo de edificio (m²)')
ax.set_ylabel('Área (m²)')
ax.set_ylim(0, 3500)
ax.grid(True, axis='y', alpha=0.4)
savefig('techos_boxplot_area_tipo')

In [ ]:
# ── Gráfico 4.3 · Scatter: área vs volumen vegetal, coloreado por tipo ─────────
fig, ax = plt.subplots(figsize=(11, 7))

for tipo, color in zip(techos_raw['tipo_es'].unique(), MIXED):
    subset = techos_raw[techos_raw['tipo_es']==tipo]
    ax.scatter(subset['area'], subset['UN_VOL_AV'], color=color,
               alpha=0.35, s=12, label=tipo)

# Línea de regresión global
mask = techos_raw['area'].notna() & techos_raw['UN_VOL_AV'].notna()
x_fit = techos_raw.loc[mask,'area']
y_fit = techos_raw.loc[mask,'UN_VOL_AV']
z = np.polyfit(x_fit, y_fit, 1)
p = np.poly1d(z)
xs = np.linspace(x_fit.min(), x_fit.quantile(0.95), 100)
ax.plot(xs, p(xs), color='#e8f5e8', lw=1.5, ls='--', alpha=0.6, label='Reg. lineal')

ax.set_xlabel('Área del techo (m²)')
ax.set_ylabel('Volumen vegetal (UN_VOL_AV)')
ax.set_title('Área vs Volumen vegetal de techos verdes')
ax.set_xlim(0, techos_raw['area'].quantile(0.97))
ax.set_ylim(0, techos_raw['UN_VOL_AV'].quantile(0.97))
ax.legend(fontsize=8, loc='upper left')
ax.grid(True, alpha=0.2)
savefig('techos_scatter_area_volumen')

In [ ]:
# ── Gráfico 4.4 · Barras: área promedio y vol. promedio por tipo ───────────────
stats = techos_raw.groupby('tipo_es').agg(
    area_mean=('area','mean'),
    vol_mean=('UN_VOL_AV','mean'),
    count=('area','count')
).sort_values('area_mean', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Área y volumen vegetal promedio por tipo de edificio', color='#e8f5e8')

ax1.barh(stats.index, stats['area_mean'], color=GREENS[:len(stats)], edgecolor='#0f1a0f', height=0.6)
ax1.set_xlabel('Área promedio (m²)'); ax1.set_title('Área promedio del techo verde')
ax1.grid(True, axis='x', alpha=0.4)

ax2.barh(stats.index, stats['vol_mean'], color=WARM[:len(stats)], edgecolor='#0f1a0f', height=0.6)
ax2.set_xlabel('Volumen vegetal promedio'); ax2.set_title('Volumen vegetal promedio')
ax2.grid(True, axis='x', alpha=0.4)

savefig('techos_avg_area_vol')

In [ ]:
# ── Gráfico 4.5 · Histograma: distribución de área de techos ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(techos_raw['area'].dropna(), bins=60, color=GREENS[0],
             edgecolor='#0f1a0f', alpha=0.8)
axes[0].set_xlim(0, 2000)
axes[0].set_xlabel('Área (m²)'); axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de área de techos verdes (0–2000 m²)')
axes[0].axvline(techos_raw['area'].median(), color=ACCENT, lw=1.5, ls='--',
                label=f'Mediana: {techos_raw["area"].median():.0f} m²')
axes[0].legend(); axes[0].grid(True, axis='y', alpha=0.4)

axes[1].hist(np.log1p(techos_raw['area'].dropna()), bins=50, color=GREENS[1],
             edgecolor='#0f1a0f', alpha=0.8)
axes[1].set_xlabel('log(Área + 1)'); axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de área (escala logarítmica)')
axes[1].grid(True, axis='y', alpha=0.4)

savefig('techos_histograma_area')

## 🚫 Sección 5 — Zonas sin Gas Metano

In [ ]:
# ── Preparar datos metano ─────────────────────────────────────────────────────
metano_raw['MUNICIPIO'] = metano_raw['MUNICIPIO'].astype(str)
print(metano_raw.head(3))
print(metano_raw.shape)

In [ ]:
# ── Gráfico 5.1 · Barras: zonas sin metano por municipio ──────────────────────
by_mun = metano_raw['MUNICIPIO'].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(by_mun.index, by_mun.values, color=GREENS[:len(by_mun)],
              edgecolor='#0f1a0f', width=0.6)

for bar, val in zip(bars, by_mun.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', fontsize=9, color='#e8f5e8')

ax.set_title('Zonas sin gas metano por municipio de Milán')
ax.set_xlabel('Municipio'); ax.set_ylabel('N° de zonas')
ax.grid(True, axis='y', alpha=0.4)
savefig('metano_barras_municipio')

In [ ]:
# ── Gráfico 5.2 · Top 15 NIL (barrios) con más zonas sin metano ───────────────
by_nil = metano_raw['NIL'].value_counts().nlargest(15)

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(by_nil.index[::-1], by_nil.values[::-1],
        color=BLUES[::-1][:15], edgecolor='#0f1a0f', height=0.65)

for i, (idx, val) in enumerate(zip(by_nil.index[::-1], by_nil.values[::-1])):
    ax.text(val + 0.3, i, str(val), va='center', fontsize=9, color='#e8f5e8')

ax.set_title('Top 15 barrios (NIL) con más zonas sin gas metano')
ax.set_xlabel('N° de zonas')
ax.grid(True, axis='x', alpha=0.4)
savefig('metano_top15_nil')

In [ ]:
# ── Gráfico 5.3 · Donut: distribución por CAP (código postal) ─────────────────
by_cap = metano_raw['CAP'].value_counts().nlargest(10)
otros = metano_raw['CAP'].value_counts()[10:].sum()
by_cap_plot = pd.concat([by_cap, pd.Series({'Otros': otros})])

fig, ax = plt.subplots(figsize=(9, 7))
wedges, _, autotexts = ax.pie(
    by_cap_plot.values, labels=None,
    autopct=lambda p: f'{p:.1f}%' if p > 3 else '',
    colors=BLUES + ['#334155'],
    startangle=90, pctdistance=0.78,
    wedgeprops=dict(width=0.5, edgecolor='#0f1a0f', lw=1.5))

for at in autotexts: at.set(color='#0f1a0f', fontsize=9, fontweight='bold')
ax.legend(wedges, by_cap_plot.index, loc='lower right', fontsize=9,
          bbox_to_anchor=(1.25, 0))
ax.set_title('Distribución de zonas sin metano por Código Postal (Top 10)')
savefig('metano_donut_cap')

In [ ]:
# ── Gráfico 5.4 · Scatter geoespacial: ubicación de zonas sin metano ──────────
fig, ax = plt.subplots(figsize=(9, 9))
ax.set_facecolor('#0d1a0d')
fig.patch.set_facecolor('#0d1a0d')

municipios_uniq = metano_raw['MUNICIPIO'].unique()
color_map = {m: MIXED[i % len(MIXED)] for i, m in enumerate(sorted(municipios_uniq))}

for mun in sorted(municipios_uniq):
    sub = metano_raw[metano_raw['MUNICIPIO']==mun]
    ax.scatter(sub['LONG_WGS84'], sub['LAT_WGS84'],
               color=color_map[mun], s=22, alpha=0.7,
               label=f'Municipio {mun}', edgecolors='none')

ax.set_title('Distribución geoespacial de zonas sin gas metano\n(Milán)', color='#e8f5e8')
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
ax.legend(fontsize=8, loc='lower right', markerscale=1.5)
ax.grid(color='#1e2e1e', alpha=0.4)
savefig('metano_scatter_geo')

In [ ]:
# ── Gráfico 5.5 · Mapa de calor 2D: densidad geoespacial ──────────────────────
fig, ax = plt.subplots(figsize=(9, 8))
ax.set_facecolor('#0d1a0d')

h = ax.hist2d(metano_raw['LONG_WGS84'], metano_raw['LAT_WGS84'],
              bins=25, cmap='YlGn', alpha=0.9)
cbar = plt.colorbar(h[3], ax=ax)
cbar.set_label('Densidad de zonas', color='#6b8f6b')

ax.set_title('Densidad geoespacial: zonas sin gas metano en Milán')
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
savefig('metano_heatmap_geo')

## 🔗 Sección 6 — Análisis Cruzado

In [ ]:
# ── Gráfico 6.1 · Barras duales: m²/hab + total verde por año ─────────────────
total_mq_year = verde_raw.groupby('Anno')['Metri Quadri'].sum().reset_index()
total_mq_year.columns = ['Anno','total_mq']
total_mq_year['Anno'] = total_mq_year['Anno'].astype(int)
mq_hab2 = mq_hab[['Anno','Mq_verde_urbano_per_abitante']].copy()
merged = pd.merge(total_mq_year, mq_hab2, on='Anno', how='inner')

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.bar(merged['Anno'], merged['total_mq']/1e6, color=GREENS[2], alpha=0.6,
        label='Total m² verde (M)', width=0.6)
ax2.plot(merged['Anno'], merged['Mq_verde_urbano_per_abitante'], color=ACCENT,
         lw=2.5, marker='o', markersize=7, label='m²/hab')

ax1.set_ylabel('Total m² (millones)', color=GREENS[2])
ax2.set_ylabel('m² / habitante', color=ACCENT)
ax1.set_xlabel('Año')
ax1.set_title('Verde urbano total y por habitante (2011–2024)')
lines1, l1 = ax1.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, l1+l2)
ax1.grid(True, axis='y', alpha=0.3)
savefig('cross_verde_total_vs_habitante')

In [ ]:
# ── Gráfico 6.2 · Pair plot simplificado: variables climáticas ────────────────
vars_pair = ['temp_media','dias_calura','noches_trop','precip_mm','humedad']
fig, axes = plt.subplots(len(vars_pair), len(vars_pair), figsize=(14, 12))
fig.suptitle('Correlación entre variables climáticas por zona', color='#e8f5e8', fontsize=13)

for i, vi in enumerate(vars_pair):
    for j, vj in enumerate(vars_pair):
        ax = axes[i][j]
        if i == j:
            ax.hist(clima[vi], bins=7, color=GREENS[i], edgecolor='#0f1a0f', alpha=0.8)
        else:
            ax.scatter(clima[vj], clima[vi], color=GREENS[i], s=40, edgecolors='#e8f5e8', lw=0.4)
        if j == 0: ax.set_ylabel(vi, fontsize=8)
        if i == len(vars_pair)-1: ax.set_xlabel(vj, fontsize=8)
        ax.tick_params(labelsize=6)
        ax.grid(True, alpha=0.2)

savefig('cross_pairplot_clima')

In [ ]:
# ── Gráfico 6.3 · KPI dashboard visual ────────────────────────────────────────
fig = plt.figure(figsize=(14, 7))
fig.suptitle('Dashboard KPI — Milán Verde & Sostenible', fontsize=16, color='#e8f5e8', y=0.98)

kpis = [
    ('Verde / hab. (2022)', '18.9 m²', '+9.9% vs 2011', GREENS[0]),
    ('Total verde urbano', '25.7M m²', '+18% vs 2011', GREENS[1]),
    ('Techos verdes', '4.396', '89% residencial', GREENS[2]),
    ('Reducción gas', '−25.8%', '509→377.9 m³/hab', BLUES[0]),
    ('Reducción agua', '−9.8%', '92.1→83.1 m³/hab', BLUES[1]),
    ('Temp. máx (Centro)', '38.2°C', 'Zona más cálida', WARM[0]),
    ('Noches tropicales', '63.3/año', 'Bocconi, máximo', RED),
    ('Zonas sin metano', '366', '9 municipios', WARM[2]),
]

for i, (label, val, sub, color) in enumerate(kpis):
    ax = fig.add_subplot(2, 4, i+1)
    ax.set_facecolor('#111811')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0,0),1,1, facecolor='#0a150a', edgecolor=color, lw=1.5, transform=ax.transAxes))
    ax.text(0.5, 0.75, val, ha='center', va='center', fontsize=20, fontweight='bold',
            color=color, transform=ax.transAxes)
    ax.text(0.5, 0.45, label, ha='center', va='center', fontsize=9,
            color='#e8f5e8', transform=ax.transAxes)
    ax.text(0.5, 0.18, sub, ha='center', va='center', fontsize=8,
            color='#6b8f6b', transform=ax.transAxes)

plt.tight_layout()
savefig('cross_kpi_dashboard')

In [ ]:
# ── Gráfico 6.4 · Correlación: matriz de variables climáticas ──────────────────
corr_matrix = clima[['temp_media','temp_max','temp_min','dias_calura',
                       'noches_trop','dias_gelo','humedad','precip_mm',
                       'dias_lluvia','viento_ms']].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, ax=ax, cmap='RdYlGn', center=0,
            annot=True, fmt='.2f', annot_kws={'size':9},
            linewidths=0.5, linecolor='#0f1a0f',
            cbar_kws={'label': 'Correlación de Pearson'})

ax.set_title('Matriz de correlación — Variables climáticas de Milán')
savefig('cross_correlacion_matriz')

---
## ✅ Fin del análisis
**Total de gráficos generados: 30**

| Sección | Gráficos |
|---|---|
| Verde Urbano | 6 |
| Consumo de Recursos | 5 |
| Clima por Zona | 5 |
| Techos Verdes | 5 |
| Zonas sin Metano | 5 |
| Análisis Cruzado | 4 |

> **Nota:** Los CSVs deben estar en la misma carpeta que este notebook.